In [1]:
from kLLMmeans import kLLMmeans, get_embeddings, summarize_cluster, spherical_kmeans_fit_predict
from experiment_utils import load_dataset, cluster_metrics, avg_closest_distance
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering
from sklearn_extra.cluster import KMedoids
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

import numpy as np
import pandas as pd
import json
import pickle
import os

import warnings
warnings.filterwarnings("ignore")

# --- Config: no-API baseline path (Bank77) ---
DATASETS = ["bank77", "clinic", "goemo", "massive_D", "massive_I"]
# Keys must exist in processed_data/data_<dataset>.pkl under ['embeddings'].
# openai: Azure (AZURE_OPENAI_* + AZURE_OPENAI_EMBEDDING_DEPLOYMENT) or OPENAI_KEY.
EMBEDDING_TYPES = ["distilbert", "e5-large", "sbert", "openai"]
RUN_KMEDOIDS = True
RUN_BROKEN_BASELINES = False  # GMM / spectral / agglomerative (uses StandardScaler + PCA)
RUN_LLM = False
NUM_SEEDS = 10

In [2]:
max_iter = 120

# baseline_results[data][emb_type][seed][method] -> dict with assignments, centroids, results (list)
baseline_results = {}

for data in DATASETS:
    baseline_results[data] = {}
    pkl_path = os.path.join("processed_data", f"data_{data}.pkl")
    with open(pkl_path, "rb") as f:
        data_dict = pickle.load(f)

    labels = data_dict["labels"]
    num_clusters = data_dict["num_clusters"]
    documents = data_dict["documents"]
    text_features = data_dict["embeddings"]
    prompt = data_dict["prompt"]
    text_type = data_dict["text_type"]
    oracle_cluster_assignments = labels

    for emb_type in EMBEDDING_TYPES:
        if emb_type not in text_features:
            print(f"Skipping {data}: missing embedding key {emb_type!r}")
            continue

        baseline_results[data][emb_type] = {}
        emb_data = text_features[emb_type]

        oracle_clustered_embeddings = {i: [] for i in range(num_clusters)}
        for embedding, cluster in zip(emb_data, oracle_cluster_assignments):
            oracle_clustered_embeddings[cluster].append(embedding)
        oracle_centroids = [
            np.mean(oracle_clustered_embeddings[i], axis=0) if oracle_clustered_embeddings[i] else None
            for i in range(num_clusters)
        ]
        oracle_summary_embeddings = oracle_centroids

        for seed in range(NUM_SEEDS):
            baseline_results[data][emb_type][seed] = {}

            # Euclidean k-means
            kmeans = KMeans(n_clusters=num_clusters, max_iter=max_iter, random_state=seed)
            kmeans_assignments = kmeans.fit_predict(emb_data)
            kmeans_centroids = kmeans.cluster_centers_
            results = cluster_metrics(
                np.array(labels), kmeans_assignments, oracle_centroids, kmeans_centroids, oracle_summary_embeddings
            )
            baseline_results[data][emb_type][seed]["kmeans"] = {
                "assignments": kmeans_assignments,
                "final_centroids": kmeans_centroids,
                "results": results,
            }
            print([data, emb_type, seed, "kmeans", results])

            # Spherical k-means (not reported in Diaz-Rodriguez 2025 for Bank77)
            kmeans_spherical_assignments, kmeans_spherical_centroids = spherical_kmeans_fit_predict(
                emb_data, num_clusters, max_iter=max_iter, random_state=seed, normalize=True
            )
            results = cluster_metrics(
                np.array(labels),
                kmeans_spherical_assignments,
                oracle_centroids,
                kmeans_spherical_centroids,
                oracle_summary_embeddings,
            )
            baseline_results[data][emb_type][seed]["kmeans_spherical"] = {
                "assignments": kmeans_spherical_assignments,
                "final_centroids": kmeans_spherical_centroids,
                "results": results,
            }
            print([data, emb_type, seed, "kmeans_spherical", results])

            if RUN_KMEDOIDS:
                kmedoids = KMedoids(n_clusters=num_clusters, max_iter=max_iter, random_state=seed)
                kmedoids.fit(emb_data)
                kmedoids_assignments = kmedoids.labels_
                kmedoids_indices = kmedoids.medoid_indices_
                kmedoids_centroids = emb_data[kmedoids_indices]
                results = cluster_metrics(
                    np.array(labels),
                    kmedoids_assignments,
                    oracle_centroids,
                    kmedoids_centroids,
                    oracle_summary_embeddings,
                )
                baseline_results[data][emb_type][seed]["kmedoids"] = {
                    "assignments": kmedoids_assignments,
                    "final_centroids": kmedoids_centroids,
                    "results": results,
                }
                print([data, emb_type, seed, "kmedoids", results])

            if RUN_BROKEN_BASELINES:
                X_scaled = StandardScaler().fit_transform(emb_data)

                pca = PCA(n_components=50, random_state=seed)
                X_pca = pca.fit_transform(X_scaled)
                gmm = GaussianMixture(n_components=num_clusters, random_state=seed)
                gmm_assignments = gmm.fit_predict(X_pca)
                gmm_clustered_embeddings = {i: [] for i in range(num_clusters)}
                for embedding, cluster in zip(emb_data, gmm_assignments):
                    gmm_clustered_embeddings[cluster].append(embedding)
                gmm_centroids = [
                    np.mean(gmm_clustered_embeddings[i], axis=0) if gmm_clustered_embeddings[i] else None
                    for i in range(num_clusters)
                ]
                results = cluster_metrics(
                    np.array(labels), gmm_assignments, gmm_centroids, oracle_centroids, oracle_centroids
                )
                baseline_results[data][emb_type][seed]["gmm"] = {
                    "assignments": gmm_assignments,
                    "results": results,
                }
                print([data, emb_type, seed, "gmm", results])

                spectral = SpectralClustering(
                    n_clusters=num_clusters, random_state=seed, affinity="nearest_neighbors"
                )
                spectral_assignments = spectral.fit_predict(X_scaled)
                spectral_clustered_embeddings = {i: [] for i in range(num_clusters)}
                for embedding, cluster in zip(emb_data, spectral_assignments):
                    spectral_clustered_embeddings[cluster].append(embedding)
                spectral_centroids = [
                    np.mean(spectral_clustered_embeddings[i], axis=0)
                    if spectral_clustered_embeddings[i]
                    else None
                    for i in range(num_clusters)
                ]
                results = cluster_metrics(
                    np.array(labels),
                    spectral_assignments,
                    spectral_centroids,
                    oracle_centroids,
                    oracle_centroids,
                )
                baseline_results[data][emb_type][seed]["spectral"] = {
                    "assignments": spectral_assignments,
                    "results": results,
                }
                print([data, emb_type, seed, "spectral", results])

                agglo = AgglomerativeClustering(n_clusters=num_clusters)
                agglo_assignments = agglo.fit_predict(X_scaled)
                agglo_clustered_embeddings = {i: [] for i in range(num_clusters)}
                for embedding, cluster in zip(emb_data, agglo_assignments):
                    agglo_clustered_embeddings[cluster].append(embedding)
                agglo_centroids = [
                    np.mean(agglo_clustered_embeddings[i], axis=0)
                    if agglo_clustered_embeddings[i]
                    else None
                    for i in range(num_clusters)
                ]
                results = cluster_metrics(
                    np.array(labels),
                    agglo_assignments,
                    agglo_centroids,
                    oracle_centroids,
                    oracle_centroids,
                )
                baseline_results[data][emb_type][seed]["agglomerative"] = {
                    "assignments": agglo_assignments,
                    "results": results,
                }
                print([data, emb_type, seed, "agglomerative", results])

            if RUN_LLM:
                for llm_type in [
                    "gpt-3.5-turbo",
                    "gpt-4o",
                    "llama3.3-70b",
                    "deepseek-chat",
                    "claude-3-7-sonnet-20250219",
                ]:
                    baseline_results[data][emb_type][seed][llm_type] = {}
                    for force_context_length in [0, 10]:
                        baseline_results[data][emb_type][seed][llm_type][force_context_length] = {}
                        for max_llm_iter in [1, 5]:
                            (
                                assignments,
                                final_summaries,
                                final_summary_embeddings,
                                final_centroids,
                                summaries_evolution,
                                centroids_evolution,
                            ) = kLLMmeans(
                                documents,
                                prompt=prompt,
                                text_type=text_type,
                                num_clusters=num_clusters,
                                force_context_length=force_context_length,
                                max_llm_iter=max_llm_iter,
                                max_iter=max_iter,
                                tol=1e-4,
                                random_state=seed,
                                emb_type=emb_type,
                                text_features=text_features[emb_type],
                            )
                            results = cluster_metrics(
                                np.array(labels),
                                assignments,
                                oracle_centroids,
                                final_centroids,
                                oracle_summary_embeddings,
                                final_summary_embeddings,
                            )
                            baseline_results[data][emb_type][seed][llm_type][force_context_length][
                                max_llm_iter
                            ] = {
                                "assignments": assignments,
                                "final_summaries": final_summaries,
                                "final_summary_embeddings": final_summary_embeddings,
                                "final_centroids": final_centroids,
                                "summaries_evolution": summaries_evolution,
                                "centroids_evolution": centroids_evolution,
                                "results": results,
                            }
                            print(
                                [
                                    data,
                                    emb_type,
                                    llm_type,
                                    seed,
                                    force_context_length,
                                    max_llm_iter,
                                    results,
                                ]
                            )

    os.makedirs("results", exist_ok=True)
    out_path = os.path.join("results", f"sims_offline_{data}_baseline.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(baseline_results[data], f)
    print(f"Saved {out_path}")


['bank77', 'distilbert', 0, 'kmeans', [0.44155844155844154, 0.6423646258215746, 0.33949136932089075, 0.33949136932089075, 0.33949136932089075, 0.33949136932089075]]
['bank77', 'distilbert', 0, 'kmeans_spherical', [0.44415584415584414, 0.645080300435098, 0.45547965913007155, 0.45547965913007155, 0.45547965913007155, 0.45547965913007155]]
['bank77', 'distilbert', 0, 'kmedoids', [0.3288961038961039, 0.5599096170519267, 0.590943414262749, 0.590943414262749, 0.590943414262749, 0.590943414262749]]
['bank77', 'distilbert', 1, 'kmeans', [0.42954545454545456, 0.645466790147295, 0.3414311702910561, 0.3414311702910561, 0.3414311702910561, 0.3414311702910561]]
['bank77', 'distilbert', 1, 'kmeans_spherical', [0.42857142857142855, 0.644495334653429, 0.45810389043633226, 0.45810389043633226, 0.45810389043633226, 0.45810389043633226]]
['bank77', 'distilbert', 1, 'kmedoids', [0.3288961038961039, 0.5599096170519267, 0.590943414262749, 0.590943414262749, 0.590943414262749, 0.590943414262749]]
['bank77', 

In [3]:
# Diaz-Rodriguez et al. 2025, arXiv:2502.09667v4, Table 5.
# Bank77, text-embedding-3-small, 10 seeds.  Values are ACC (std) and NMI (std).
# Tables 2/6 compare embeddings on other datasets — no Bank77 paper row for distilbert / e5-large / sbert.
# Spherical k-means is not reported in the paper.
PAPER_BANK77 = {
    "openai": {
        "kmeans":        {"acc_mean": 66.2, "acc_std": 1.82, "nmi_mean": 83.0, "nmi_std": 0.56},
        "kmedoids":      {"acc_mean": 41.7, "acc_std": 0.0,  "nmi_mean": 69.5, "nmi_std": 0.0},
        "gmm":           {"acc_mean": 67.7, "acc_std": 1.78, "nmi_mean": 83.0, "nmi_std": 0.54},
        "agglomerative": {"acc_mean": 69.9, "acc_std": 0.0,  "nmi_mean": 83.7, "nmi_std": 0.0},
        "spectral":      {"acc_mean": 68.2, "acc_std": 0.56, "nmi_mean": 83.3, "nmi_std": 0.38},
    },
}


def _load_bank77_baseline_if_needed():
    """Use in-memory baseline_results from cell 1, or load results/sims_offline_bank77_baseline.pkl."""
    br = globals().get("baseline_results")
    if isinstance(br, dict) and br.get("bank77"):
        return br["bank77"]
    path = os.path.join("results", "sims_offline_bank77_baseline.pkl")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Run Cell 1 first, or place a previously saved baseline pickle at that path."
        )
    with open(path, "rb") as f:
        return pickle.load(f)


def build_bank77_paper_comparison_table(per_dataset_results):
    """
    cluster_metrics returns ACC and NMI in [0, 1]; paper reports percentages (x100).
    """
    rows = []
    methods = ["kmeans", "kmeans_spherical"]
    if RUN_KMEDOIDS:
        methods.append("kmedoids")

    for emb_type, seed_dict in sorted(per_dataset_results.items()):
        if not seed_dict:
            continue
        for method in methods:
            if not any(method in seed_dict.get(s, {}) for s in seed_dict):
                continue
            accs = []
            nmis = []
            for seed in range(NUM_SEEDS):
                if seed not in seed_dict or method not in seed_dict[seed]:
                    continue
                r = seed_dict[seed][method]["results"]
                accs.append(float(r[0]) * 100.0)
                nmis.append(float(r[1]) * 100.0)
            if not accs:
                continue

            acc_mean = float(np.mean(accs))
            acc_std = float(np.std(accs, ddof=1)) if len(accs) > 1 else 0.0
            nmi_mean = float(np.mean(nmis))
            nmi_std = float(np.std(nmis, ddof=1)) if len(nmis) > 1 else 0.0

            paper = PAPER_BANK77.get(emb_type, {}).get(method)
            if paper is None:
                paper_acc_m = paper_nmi_m = np.nan
                paper_acc_s = paper_nmi_s = np.nan
                delta_acc = np.nan
            else:
                paper_acc_m = paper["acc_mean"]
                paper_acc_s = paper["acc_std"]
                paper_nmi_m = paper["nmi_mean"]
                paper_nmi_s = paper["nmi_std"]
                delta_acc = acc_mean - paper_acc_m

            rows.append(
                {
                    "dataset": "bank77",
                    "emb_type": emb_type,
                    "method": method,
                    "acc_mean": acc_mean,
                    "acc_std": acc_std,
                    "nmi_mean": nmi_mean,
                    "nmi_std": nmi_std,
                    "paper_acc_mean": paper_acc_m,
                    "paper_acc_std": paper_acc_s,
                    "paper_nmi_mean": paper_nmi_m,
                    "paper_nmi_std": paper_nmi_s,
                    "delta_acc_vs_paper": delta_acc,
                }
            )
    return pd.DataFrame(rows)


bank77_runs = _load_bank77_baseline_if_needed()
comparison_df = build_bank77_paper_comparison_table(bank77_runs)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
comparison_df


,dataset,emb_type,method,acc_mean,acc_std,nmi_mean,nmi_std,paper_acc_mean,paper_acc_std,paper_nmi_mean,paper_nmi_std,delta_acc_vs_paper
0,bank77,distilbert,kmeans,43.675325,7.505550e-01,64.440409,4.110788e-01,NaN,NaN,NaN,NaN,NaN
1,bank77,distilbert,kmeans_spherical,43.987013,1.189472e+00,64.410961,4.661141e-01,NaN,NaN,NaN,NaN,NaN
2,bank77,distilbert,kmedoids,32.889610,7.489778e-15,55.990962,7.489778e-15,NaN,NaN,NaN,NaN,NaN
3,bank77,e5-large,kmeans,58.412338,2.240780e+00,76.799113,9.485360e-01,NaN,NaN,NaN,NaN,NaN
4,bank77,e5-large,kmeans_spherical,58.246753,1.752846e+00,76.759395,8.450156e-01,NaN,NaN,NaN,NaN,NaN
5,bank77,e5-large,kmedoids,37.175325,7.489778e-15,64.486751,0.000000e+00,NaN,NaN,NaN,NaN,NaN
6,bank77,openai,kmeans,65.918831,1.311539e+00,82.881021,4.863760e-01,66.2,1.82,83.0,0.56,-0.281169
7,bank77,openai,kmeans_spherical,66.334416,1.207227e+00,83.088871,4.792456e-01,NaN,NaN,NaN,NaN,NaN
8,bank77,openai,kmedoids,41.136364,0.000000e+00,69.335688,0.000000e+00,41.7,0.00,69.5,0.00,-0.563636
9,bank77,sbert,kmeans,63.594156,2.319810e+00,80.994885,8.936653e-01,NaN,NaN,NaN,NaN,NaN
